In [3]:
#Setup e Caricamento del File

!pip install pandas scipy

from google.colab import files
from google.colab import output
from google.colab import drive
import json
import pandas as pd
import scipy.stats as st
import numpy as np
import io

drive.mount('/content/drive')
FILE_PATH_SU_DRIVE = "/content/drive/My Drive/Colab Notebooks/risultatiSimulazione/provaVec.json"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Analisi Statistica "Memory-Efficient" (Metrica 4)

#Implementa la metodologia del file stat copia.pdf, Sezione 6.3.3b:
#Calcola p_m (la proporzione sopra soglia) per ogni singola replica.

with open(FILE_PATH_SU_DRIVE, 'r') as f:
    data = json.load(f)

risultati_per_replica = []

PARAM_MAP = {
    '$0': 'm_SS1', '$1': 'm_SS2', '$2': 'm_arrivi',
    '$3': 'q_prob_U1', '$4': 'p_feedback'
}
THRESHOLDS = [2.0, 3.0, 4.0] #Soglie da var-8.pdf

for run_id, run_data in data.items():
    attributes = run_data.get('attributes', {})
    config_name = attributes.get('experiment', 'N/A')

    try:
        repetition = int(attributes.get('repetition', -1))
    except ValueError:
        repetition = -1

    itervars_str = attributes.get('iterationvars', '')
    params = {}
    if itervars_str:
        try:
            pairs = itervars_str.split(', ')
            for pair in pairs:
                key, value = pair.split('=')
                col_name = PARAM_MAP.get(key, key)
                params[col_name] = value.replace('s', '')
        except Exception:
            pass

    row = {
        'Config': config_name,
        'repetition': repetition
    }
    row.update(params)

    vectors = run_data.get('vectors', [])

    if not vectors:
        continue

    for vec in vectors:
        metric_name = vec.get('name')

        if not (metric_name and metric_name.startswith('sojournTime')):
            continue

        raw_data = vec.get('value', [])

        values = []
        if raw_data:
            # Solo i valori (indici dispari)
            values = [float(v) for v in raw_data[1::2]]

        total_count = len(values) # N_valid (N - n0) nella Eq. 6.104

        #Calcolo Metrica 4 (Proporzioni)
        #Implementa Eq. 6.104 (stat copia.pdf)
        if total_count == 0:
            for t in THRESHOLDS:
                #Se ho starvation, il tempo è infinito -> Sicuramente > Soglia
                # Quindi la probabilità è 1.0 (100%)
                row[f"{metric_name}_P(T > {t}s)"] = 1.0
        else:
            for t in THRESHOLDS:
                #Calcola v_m (Eq. 6.105): quanti campioni > threshold
                threshold_count = sum(1 for v in values if v > t)

                #Calcola p_m (Eq. 6.104): proporzione campionaria per la replica
                p_m = threshold_count / total_count
                row[f"{metric_name}_P(T > {t}s)"] = p_m

    risultati_per_replica.append(row)

# 3. Crea il DataFrame Pandas finale
df_risultati = pd.DataFrame(risultati_per_replica)

display(df_risultati.head())

,Config,repetition,m_SS1,m_SS2,m_arrivi,q_prob_U1,p_feedback,sojournTimeU1_vec:vector_P(T > 2.0s),sojournTimeU1_vec:vector_P(T > 3.0s),sojournTimeU1_vec:vector_P(T > 4.0s),sojournTimeU2_vec:vector_P(T > 2.0s),sojournTimeU2_vec:vector_P(T > 3.0s),sojournTimeU2_vec:vector_P(T > 4.0s)
0,Config1,7,2.4,4.0,4.0,0.4,0.8,0.999503,0.997016,0.987071,1.0,1.0,1.0
1,Config1,8,2.4,4.0,4.0,0.4,0.8,1.000000,1.000000,1.000000,1.0,1.0,1.0
2,Config1,9,2.4,4.0,4.0,0.4,0.8,0.998984,0.998476,0.994919,1.0,1.0,1.0
3,Config1,0,2.4,4.0,4.0,0.4,0.9,1.000000,1.000000,1.000000,1.0,1.0,1.0
4,Config1,1,2.4,4.0,4.0,0.4,0.9,1.000000,1.000000,1.000000,1.0,1.0,1.0


In [5]:
#Calcolo Statistiche Finali (Stima Puntuale e CI)
#Calcola della media (Stima Puntuale) e del CI sui valori di proporzione (p_m)

#Definizione delle colonne di configurazione
parametri_configurazione = ['Config', 'm_SS1', 'm_SS2', 'm_arrivi', 'q_prob_U1', 'p_feedback']

#1.Raggruppamento per le 486 configurazioni uniche
grouped = df_risultati.groupby(parametri_configurazione)

#2.Calcolo delle statistiche (Mean, StdDev, Count)
stima_puntuale = grouped.mean(numeric_only=True) #Stima Puntuale (p_hat)
deviazione_standard = grouped.std(numeric_only=True)
n_campioni = grouped.count()

#3. Calcolo Intervallo di Confidenza Dinamico
#I gradi di libertà variano: dof = N - 1
degrees_of_freedom = n_campioni - 1

#Valore critico 't' per il 95% di confidenza con (n=20 -> df=19)
# Calcolo T-Value variabile (se dof <= 0 mette NaN)
t_values = np.where(
    degrees_of_freedom > 0,
    st.t.ppf(0.975, degrees_of_freedom),
    np.nan
)

margine_errore_ci = t_values * (deviazione_standard / np.sqrt(n_campioni))

ci_limite_inferiore = stima_puntuale - margine_errore_ci
ci_limite_superiore = stima_puntuale + margine_errore_ci

stima_puntuale = stima_puntuale.drop(columns='repetition')
ci_limite_inferiore = ci_limite_inferiore.drop(columns='repetition')
ci_limite_superiore = ci_limite_superiore.drop(columns='repetition')

#Stima Puntuale (Media delle 20 proporzioni p_m)
display(stima_puntuale)

#Intervallo di Confidenza (95%) - Limite Inferiore
display(ci_limite_inferiore)

#Intervallo di Confidenza (95%) - Limite Superiore
display(ci_limite_superiore)

sojournTimeU1_vec:vector_P(T > 2.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.987670   
                                       0.8                                     0.999821   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.993798   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     1.000000   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU1_vec:vector_P(T > 3.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.909521   
                                       0.8                                     0.999033   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.953176   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     0.999962   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU1_vec:vector_P(T > 4.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.810330   
                                       0.8                                     0.997473   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.894849   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     0.999716   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU2_vec:vector_P(T > 2.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                          1.0   
                                       0.8                                          1.0   
                                       0.9                                          1.0   
                             0.6       0.6  

sojournTimeU1_vec:vector_P(T > 2.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.986714   
                                       0.8                                     0.999642   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.993077   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     1.000000   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU1_vec:vector_P(T > 3.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.907258   
                                       0.8                                     0.998410   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.950966   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     0.999908   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU1_vec:vector_P(T > 4.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.806195   
                                       0.8                                     0.995714   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.891470   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     0.999495   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU2_vec:vector_P(T > 2.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                          1.0   
                                       0.8                                          1.0   
                                       0.9                                          1.0   
                             0.6       0.6  

sojournTimeU1_vec:vector_P(T > 2.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.988627   
                                       0.8                                     0.999999   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.994519   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     1.000000   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU1_vec:vector_P(T > 3.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.911784   
                                       0.8                                     0.999655   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.955386   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     1.000017   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU1_vec:vector_P(T > 4.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                     0.814466   
                                       0.8                                     0.999232   
                                       0.9                                     1.000000   
                             0.6       0.6                                     0.898228   
                                       0.8                                     1.000000   
...                                                                                 ...   
Config2 3.5   4.0   6.0      0.6       0.8                                     1.000000   
                                       0.9                                     1.000000   
                             0.8       0.6                                     0.999938   
                                       0.8                                     1.000000   
                                       0.9                                     1.000000   

                                                   sojournTimeU2_vec:vector_P(T > 2.0s)  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                                         
Config1 2.4   2.0   4.0      0.4       0.6                                          1.0   
                                       0.8                                          1.0   
                                       0.9                                          1.0   
                             0.6       0.6  

In [6]:
stima_puntuale.to_csv("metrica4_stima_puntuale.csv")
ci_limite_inferiore.to_csv("metrica4_ci_inferiore.csv")
ci_limite_superiore.to_csv("metrica4_ci_superiore.csv")

files.download("metrica4_stima_puntuale.csv")
files.download("metrica4_ci_inferiore.csv")
files.download("metrica4_ci_superiore.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>